In [1]:
import pandas as pd 
import numpy as np
import yfinance as yf
import statsmodels.api as sm

# --- CONFIGURAÇÃO ---
years = range(2015, 2025)
ks = [1, 10, 30]
modes = ["fast", "medium", "slow"]
centralities = ["central", "peripheral"]

# Função que calcula o beta
def beta_ols(rp, rm):
    # Garante que rm seja uma Series com nome
    rm = rm.squeeze().rename("rm")
    df = rp.to_frame("rp").join(rm, how="inner").dropna()
    
    if len(df) < 5: return np.nan # Proteção contra dados insuficientes
    
    y = df["rp"]
    X = sm.add_constant(df["rm"])
    model = sm.OLS(y, X).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": 5}
    )
    return model.params["rm"]

# Dicionário para armazenar os betas finais ou retornos
# Estrutura: results[year][k][centrality]
all_betas = {}

for year in years:
    print(f"Processando ano: {year}...")
    all_betas[year] = {}
    
    # 1. Baixando Benchmark (SP500) para o ano específico
    # Usamos start/end cobrindo o ano para bater com os retornos dos portfolios
    sp500 = yf.download(tickers="^GSPC", start=f"{year}-01-01", end=f"{year}-12-31", progress=False)
    sp500_dr = sp500["Close"].pct_change().dropna()
    
    # 2. Carregando retornos do ano
    try:
        returns = pd.read_parquet(f"../../data/02_clean/returns_new_{year}.parquet")
    except FileNotFoundError:
        print(f"Arquivo de retornos para {year} não encontrado. Pulando...")
        continue

    for k in ks:
        all_betas[year][k] = {}
        
        for centrality in centralities:
            # 3. Pegando metadados dos tickers (ajustado para k variável)
            try:
                metadata_path = f"../../data/07_portfolios_metadata/{centrality}_metadata_{year}_{k}.csv"
                tickers = pd.read_csv(metadata_path)["Ticker"]
                
                # Filtrar apenas tickers que existem no arquivo de retornos
                valid_tickers = [t for t in tickers if t in returns.columns]
                portfolio_returns = returns[valid_tickers].mean(axis=1) # Retorno médio do portfólio
                
                # 4. Calcular Beta
                beta_val = beta_ols(portfolio_returns, sp500_dr)
                all_betas[year][k][centrality] = beta_val
                
            except Exception as e:
                print(f"Erro no portfólio {centrality} (k={k}, ano={year}): {e}")
                all_betas[year][k][centrality] = np.nan

# --- OPCIONAL: Converter resultados para um DataFrame longo (melhor para plotar) ---
rows = []
for year, ks_dict in all_betas.items():
    for k, cents in ks_dict.items():
        for centrality, beta in cents.items():
            rows.append({"Year": year, "k": k, "Centrality": centrality, "Beta": beta})

df_results = pd.DataFrame(rows)
print("\nProcessamento concluído.")
print(df_results.head())

Processando ano: 2015...
Processando ano: 2016...
Processando ano: 2017...
Processando ano: 2018...
Processando ano: 2019...
Processando ano: 2020...
Processando ano: 2021...
Processando ano: 2022...
Processando ano: 2023...
Processando ano: 2024...

Processamento concluído.
   Year   k  Centrality      Beta
0  2015   1     central  1.122735
1  2015   1  peripheral  0.578857
2  2015  10     central  0.980891
3  2015  10  peripheral  0.639678
4  2015  30     central  1.039241


In [2]:
df_results

,Year,k,Centrality,Beta
0,2015,1,central,1.122735
1,2015,1,peripheral,0.578857
2,2015,10,central,0.980891
3,2015,10,peripheral,0.639678
4,2015,30,central,1.039241
5,2015,30,peripheral,0.624696
6,2016,1,central,1.547606
7,2016,1,peripheral,0.465895
8,2016,10,central,1.148203
9,2016,10,peripheral,0.702152


In [4]:
df_results.to_parquet("../../data/07_portfolios_metadata/beta_df.parquet")

In [1]:
import pandas as pd

years = range(2015, 2024)
returns_dict = {}

for year in years:
    for centrality in ["central", "peripheral"]:
        for k in [1, 10, 30]:
            returns_dict[f"{centrality}_{year}_{k}"] = pd.read_csv(f"../../data/06_portfolios/{centrality}_{year}_{k}.csv", index_col="Date")

In [2]:
import yfinance as yf

momentum_dict = {}

for portfolio_name, df_returns in returns_dict.items():
    first_price = yf.download(
        tickers=list(returns_dict[portfolio_name].columns),
        start=f"{year-1}-12-29",
        end=f"{year}-01-01"
    )["Close"]

    last_price = yf.download(
        tickers=list(returns_dict[portfolio_name].columns),
        start=f"{year}-12-29",
        end=f"{year+1}-01-01"
    )["Close"]

    momentum_dict[portfolio_name] = last_price.iloc[0] / first_price.iloc[0] - 1

[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*****

KeyboardInterrupt: 

In [ ]:
momentum_df = (
    pd.concat(momentum_dict, names=["portfolio", "Ticker"])
    .reset_index(name=f"momentum12mo_{year}")
)
momentum_df.to_parquet(f"../../data/07_portfolios_metadata/momentum_df_{year}.parquet", index=False)

In [30]:
momentum_df

,portfolio,Ticker,momentum12mo_2024
0,central,AAPL,0.316343
1,central,ACWI,0.177692
2,central,ADI,0.088667
3,central,AMAT,0.017571
4,central,AMKR,-0.204909
...,...,...,...
138,peripheral,VIRC,-0.144975
139,peripheral,WFCF,-0.087085
140,peripheral,WKSP,-0.328859
141,peripheral,XOMA,0.410270


## Completo beta e momentum

In [8]:
import pandas as pd
import numpy as np
import yfinance as yf
import statsmodels.api as sm
from tqdm import tqdm

df = pd.read_parquet(f"../../data/02_clean/returns_new_{year-9}_{year}.parquet").dropna(axis=1, how='all')
tickers = list(df.columns)

def beta_ols(rp, rm):
    joined = rp.to_frame("rp").join(rm, how="inner").dropna()
    if len(joined) < 20:  # not enough observations
        return np.nan
    y = joined["rp"]
    X = sm.add_constant(joined["^GSPC"])
    model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    return model.params["^GSPC"]

def momentum_12m(prices):
    """Total return over the year (or log return)."""
    prices = prices.dropna()
    if len(prices) < 2:
        return np.nan
    return (prices.iloc[-1] / prices.iloc[0]) - 1

beta_momentum_df = {}

for year in range(2015, 2025):
    print(f"\n=== {year} ===")
    start, end = f"{year}-01-01", f"{year}-12-31"

    # Market returns
    sp500_prices = yf.download("^GSPC", start=start, end=end, progress=False)["Close"]
    sp500_dr = sp500_prices.pct_change().dropna()
    sp500_dr.name = "rm"

    # Download all tickers at once (much faster than one by one)
    raw = yf.download(tickers, start=start, end=end, progress=True)["Close"]

    records = []
    for ticker in tqdm(tickers, desc=f"Computing {year}"):
        if ticker not in raw.columns:
            continue
        prices = raw[ticker].dropna()
        returns = prices.pct_change().dropna()

        b = beta_ols(returns, sp500_dr)
        m = momentum_12m(prices)

        records.append({
            "Ticker":            ticker,
            f"beta_{year}":      b,
            f"momentum_{year}":  m,drop(columnsdrop(columns=["ABVC", "WKSP"])=["ABVC", "WKSP"])
        })

    beta_momentum_df[year] = pd.DataFrame(records)
    print(beta_momentum_df[year].describe())


=== 2015 ===


[*********************100%***********************]  399 of 399 completed
Computing 2015: 100%|██████████| 399/399 [00:01<00:00, 289.23it/s]


        beta_2015  momentum_2015
count  399.000000     399.000000
mean     0.841828      -0.030472
std      0.312135       0.252870
min     -0.475082      -0.852590
25%      0.700904      -0.160836
50%      0.884382      -0.013509
75%      1.035415       0.107764
max      1.597882       1.299732

=== 2016 ===


[*********************100%***********************]  399 of 399 completed

3 Failed downloads:
['HNI', 'WMB', 'L']: TypeError("'NoneType' object is not subscriptable")
Computing 2016: 100%|██████████| 399/399 [00:01<00:00, 363.76it/s]


        beta_2016  momentum_2016
count  396.000000     396.000000
mean     0.988753       0.299961
std      0.495104       0.369337
min     -0.248449      -0.534789
25%      0.645195       0.110544
50%      0.968068       0.241545
75%      1.298083       0.414697
max      2.967795       3.383523

=== 2017 ===


...

        beta_2022  momentum_2022
count  399.000000     399.000000
mean     0.784242      -0.026741
std      0.340014       0.364154
min      0.058029      -0.670732
25%      0.528464      -0.239545
50%      0.777326      -0.066643
75%      0.979490       0.093575
max      2.056930       3.122418

=== 2023 ===


[*********************100%***********************]  399 of 399 completed
Computing 2023: 100%|██████████| 399/399 [00:00<00:00, 451.83it/s]


        beta_2023  momentum_2023
count  399.000000     399.000000
mean     0.902305       0.189507
std      0.394004       0.356602
min      0.074844      -0.643077
25%      0.606769      -0.031911
50%      0.898913       0.123252
75%      1.139587       0.312656
max      2.402108       2.626660

=== 2024 ===


[*********************100%***********************]  399 of 399 completed

1 Failed download:
['WWW']: TypeError("'NoneType' object is not subscriptable")
Computing 2024: 100%|██████████| 399/399 [00:00<00:00, 455.68it/s]

        beta_2024  momentum_2024
count  398.000000     398.000000
mean     0.778733       0.166054
std      0.469208       0.343426
min     -0.355520      -0.626715
25%      0.439257      -0.046656
50%      0.758992       0.129882
75%      1.075986       0.295477
max      2.266266       2.140005


In [9]:
from functools import reduce

combined_df = reduce(
    lambda left, right: pd.merge(left, right, on="Ticker", how="outer"),
    beta_momentum_df.values()
)

combined_df.to_csv("../../data/07_portfolios_metadata/beta_momentum.csv")

In [10]:
combined_df

,Ticker,beta_2015,momentum_2015,beta_2016,momentum_2016,beta_2017,momentum_2017,beta_2018,momentum_2018,beta_2019,...,beta_2020,momentum_2020,beta_2021,momentum_2021,beta_2022,momentum_2022,beta_2023,momentum_2023,beta_2024,momentum_2024
0,AAME,-0.042561,0.214524,-0.070601,-0.149421,0.218475,-0.166543,0.161756,-0.263186,0.698746,...,0.291063,0.055276,0.918773,0.305849,0.212500,-0.125306,0.273730,0.023772,0.197103,-0.359132
1,AAPL,1.142266,-0.001660,1.023703,0.123843,1.380801,0.480425,1.254701,-0.079441,1.557019,...,1.123350,0.796235,1.309359,0.385507,1.305799,-0.281995,1.104514,0.547982,0.945095,0.365199
2,ABEO,0.524800,0.020588,1.872378,0.418129,1.494744,2.170000,1.425523,-0.591860,1.631742,...,1.192609,-0.498442,0.931677,-0.810651,0.969730,-0.667027,0.866669,0.751748,0.663507,0.014467
3,ABM,0.725527,0.039297,0.725991,0.505572,1.299936,-0.064241,0.891290,-0.158748,1.246547,...,1.153171,-0.000101,0.919945,0.116924,0.839910,0.106500,1.003824,0.019394,0.817781,0.175395
4,ABT,1.139447,0.029628,1.089134,-0.082767,0.910661,0.495365,1.061069,0.231381,1.056009,...,0.853692,0.266816,0.530959,0.312249,0.834889,-0.197121,0.581190,0.024566,0.182235,0.047347
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
394,WTRG,0.612510,0.157017,0.493571,0.043630,0.472345,0.340406,0.363961,-0.103020,0.160950,...,1.074468,0.036726,0.627612,0.183474,0.662024,-0.079829,0.668718,-0.201166,0.307697,-0.010034
395,WWW,1.008828,-0.401833,1.307009,0.338770,1.147355,0.468144,0.724776,0.004313,1.246309,...,1.134894,-0.085342,1.465158,-0.020952,1.360018,-0.616502,1.503138,-0.176842,NaN,NaN
396,WY,0.870893,-0.126126,1.377943,0.051049,1.034746,0.205196,0.726370,-0.356246,0.931810,...,1.688680,0.165705,1.205855,0.290458,0.976653,-0.192456,1.096529,0.185776,0.584460,-0.174260
397,XOM,1.067680,-0.129705,0.853615,0.206404,0.589197,-0.044736,0.860914,-0.165058,0.942338,...,1.094894,-0.366399,0.974871,0.554893,0.533945,0.804778,0.548500,-0.029260,0.188159,0.068461


In [15]:
metadata = pd.read_csv("../../data/02_clean/METADADOS_ATUALIZADO - Sheet1.csv")
metadata["mcap_2024"]

0      3.754818e+12
1      7.254202e+11
2      1.468970e+15
3      6.937964e+11
4      6.758384e+11
           ...     
394    2.054726e+09
395    5.491550e+09
396    5.117357e+10
397    1.722747e+11
398    1.385571e+11
Name: mcap_2024, Length: 399, dtype: float64